<a href="https://colab.research.google.com/github/souhirbenamor/EPF/blob/main/DNN_model_Final_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --upgrade tensorflow==2.12.1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
pip install gitpython

In [ ]:
import os
from git.repo.base import Repo
Repo.clone_from("https://github.com/jeslago/epftoolbox", "epftoolbox")

<git.repo.base.Repo '/content/epftoolbox/.git'>

In [ ]:
cd epftoolbox

/content/epftoolbox


In [ ]:
#type(scaler)
import pandas as pd
import numpy as np
import argparse
import os

from epftoolbox.data import read_data
from epftoolbox.evaluation import MAE, sMAPE
from epftoolbox.models import LEAR

In [ ]:
from epftoolbox.models import evaluate_lear_in_test_dataset
import os

# Market under study. If it not one of the standard ones, the file name
# has to be provided, where the file has to be a csv file
dataset = 'EPF_data'

In [ ]:
# Number of years (a year is 364 days) in the test dataset.
years_test = 2.0195

# Number of days used in the training dataset for recalibration
calibration_window = 364 * 4

# Optional parameters for selecting the test dataset, if either of them is not provided,
# the test dataset is built using the years_test parameter. They should either be one of
# the date formats existing in python or a string with the following format
# "%d/%m/%Y %H:%M"
begin_test_date = '01/01/2023'
end_test_date = '31/12/2024'


In [ ]:
path_datasets_folder = os.path.join('.', 'datasets')
path_recalibration_folder = os.path.join('.', 'experimental_files')

In [ ]:
dataset

'EPF_plus'

In [ ]:
EPF_1=pd.read_excel('/content/EPF_data_FUND25.xlsx')
EPF_1.reset_index(inplace=True)
EPF_1["Date"] = pd.to_datetime(EPF_1["Date"])
EPF_1 = EPF_1.set_index('Date')
del EPF_1['index']
EPF_1.head()
print(EPF_1)

                     Price  Demand Day-ahead DE  \
Date                                              
2019-01-01 00:00:00  10.07         41662.750000   
2019-01-01 01:00:00  -4.08         40553.000000   
2019-01-01 02:00:00  -9.91         40261.250000   
2019-01-01 03:00:00  -7.41         40603.250000   
2019-01-01 04:00:00 -12.55         40904.250000   
...                    ...                  ...   
2024-12-31 20:00:00  15.70         52497.653679   
2024-12-31 21:00:00   9.06         49201.667324   
2024-12-31 22:00:00   0.52         47257.128235   
2024-12-31 23:00:00   2.16         46727.739220   
2025-01-01 00:00:00   2.16         46727.739220   

                     Wind and PV Day ahead (MWh/h)     Gas    Coal    CO2  
Date                                                                       
2019-01-01 00:00:00                       25668.75  21.980   75.44  24.73  
2019-01-01 01:00:00                       27384.00  21.980   75.44  24.73  
2019-01-01 02:00:00             

In [ ]:
EPF_1 = pd.read_excel('/content/EPF_data_FUND25.xlsx')
EPF_1['Date'] = pd.to_datetime(EPF_1['Date'])
EPF_1 = EPF_1.set_index('Date', drop=True)

# ───› STEP A: force a complete hourly index and back‑/forward‑fill gaps
EPF_1 = EPF_1.asfreq('H')
EPF_1.fillna(method='ffill', inplace=True)
EPF_1.fillna(method='bfill', inplace=True)

# ───› STEP B: now do your inf→NaN cleanup and clamp
import numpy as np
EPF_1.replace([np.inf, -np.inf], np.nan, inplace=True)
EPF_1.dropna(how='any', inplace=True)

for col in EPF_1.columns:
    μ, σ = EPF_1[col].mean(), EPF_1[col].std()
    EPF_1[col] = EPF_1[col].clip(μ - 5*σ, μ + 5*σ)




<ipython-input-40-d065682b3561>:6: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  EPF_1 = EPF_1.asfreq('H')
<ipython-input-40-d065682b3561>:7: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  EPF_1.fillna(method='ffill', inplace=True)
<ipython-input-40-d065682b3561>:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  EPF_1.fillna(method='bfill', inplace=True)


In [ ]:
EPF_1.to_csv('/content/EPF_data_FUND25.csv')

In [ ]:
from epftoolbox.models import hyperparameter_optimizer

# Number of layers in DNN
nlayers = 2

# Market under study. If it not one of the standard ones, the file name
# has to be provided, where the file has to be a csv file
dataset = 'EPF_data_FUND25'

# Number of years (a year is 364 days) in the test dataset.
years_test = 2

# Optional parameters for selecting the test dataset, if either of them is not provided,
# the test dataset is built using the years_test parameter. They should either be one of
# the date formats existing in python or a string with the following format
# "%d/%m/%Y %H:%M"
begin_test_date = '01/01/2023'
end_test_date = '31/12/2024'
# Boolean that selects whether the validation and training datasets are shuffled
shuffle_train = 1

# Boolean that selects whether a data augmentation technique for DNNs is used
data_augmentation = 0

# Boolean that selects whether we start a new hyperparameter optimization or we restart an existing one
new_hyperopt = 1

# Number of years used in the training dataset for recalibration
calibration_window = 4

# Unique identifier to read the trials file of hyperparameter optimization
experiment_id = 1

# Number of iterations for hyperparameter optimization
max_evals = 1500

path_datasets_folder = "./datasets/"
path_hyperparameters_folder = "./experimental_files/"

In [ ]:
# Check documentation of the hyperparameter_optimizer for each of the function parameters
# In this example, we optimize a model for the PJM market.
# We consider two directories, one for storing the datasets and the other one for the experimental files.
# We start a hyperparameter optimization from scratch. We employ 1500 iterations in hyperopt,
# 2 years of test data, a DNN with 2 hidden layers, a calibration window of 4 years,
# we avoid data augmentation,  and we provide an experiment_id equal to 1
hyperparameter_optimizer(path_datasets_folder=path_datasets_folder,
                         path_hyperparameters_folder=path_hyperparameters_folder,
                         new_hyperopt=new_hyperopt, max_evals=max_evals, nlayers=nlayers, dataset=dataset,
                         years_test=years_test, calibration_window=calibration_window,
                         shuffle_train=shuffle_train, data_augmentation=0, experiment_id=experiment_id,
                         begin_test_date=begin_test_date, end_test_date=end_test_date)

Test datasets: 2023-01-01 00:00:00 - 2024-12-31 23:00:00




Tested 1/1500 iterations.
Best MAE - Validation Dataset
  MAE: 28.1 | sMAPE: 40.44 %

Best MAE - Test Dataset
  MAE: 33.6 | sMAPE: 51.99 %




Tested 2/1500 iterations.
Best MAE - Validation Dataset
  MAE: 14.4 | sMAPE: 19.35 %

Best MAE - Test Dataset
  MAE: 18.5 | sMAPE: 33.35 %




Tested 3/1500 iterations.
Best MAE - Validation Dataset
  MAE: 14.4 | sMAPE: 19.35 %

Best MAE - Test Dataset
  MAE: 18.5 | sMAPE: 33.35 %




Tested 4/1500 iterations.
Best MAE - Validation Dataset
  MAE: 14.4 | sMAPE: 19.35 %

Best MAE - Test Dataset
  MAE: 18.5 | sMAPE: 33.35 %


...



Tested 52/1500 iterations.
Best MAE - Validation Dataset
  MAE: 10.2 | sMAPE: 17.50 %

Best MAE - Test Dataset
  MAE: 21.3 | sMAPE: 35.08 %




Tested 53/1500 iterations.
Best MAE - Validation Dataset
  MAE: 10.2 | sMAPE: 17.50 %

Best MAE - Test Dataset
  MAE: 21.3 | sMAPE: 35.08 %


/content/epftoolbox/epftoolbox/data/_wrangling.py:95: RuntimeWarning: overflow encountered in sinh
  transformed_data = np.sinh(data)
/content/epftoolbox/epftoolbox/data/_wrangling.py:67: RuntimeWarning: overflow encountered in multiply
  transformed_data[:, i] = data[:, i] * self.mad[i] + self.median[i]




Tested 54/1500 iterations.
Best MAE - Validation Dataset
  MAE: 10.2 | sMAPE: 17.50 %

Best MAE - Test Dataset
  MAE: 21.3 | sMAPE: 35.08 %




Tested 55/1500 iterations.
Best MAE - Validation Dataset
  MAE: 10.2 | sMAPE: 17.50 %

Best MAE - Test Dataset
  MAE: 21.3 | sMAPE: 35.08 %
